In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [3]:
# Load and combine
df_2023 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2023_c20260323.csv')
df_2024 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2024_c20260323 2.csv')
df = pd.concat([df_2023, df_2024], ignore_index=True)

# Filter to top 10 event types
top_10_types = df['EVENT_TYPE'].value_counts().head(10).index.tolist()
print(df['EVENT_TYPE'].value_counts().head(10))
df = df[df['EVENT_TYPE'].isin(top_10_types)].copy()

# Feature engineering (all vectorized, no loops)
month_map = {'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
             'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}
df['MONTH'] = df['MONTH_NAME'].map(month_map)

# Hour from BEGIN_TIME (stored as HHMM integer, e.g. 1430 -> 14)
df['HOUR'] = df['BEGIN_TIME'] // 100

# Duration in minutes from full timestamps
df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['DURATION_MIN'] = (df['END_DT'] - df['BEGIN_DT']).dt.total_seconds() / 60

# CZ_TYPE to binary
df['CZ_TYPE_ENC'] = (df['CZ_TYPE'] == 'C').astype(int)

# Label encode state
le_state = LabelEncoder()
df['STATE_ENC'] = le_state.fit_transform(df['STATE'].astype(str))

# Label encode WFO
le_wfo = LabelEncoder()
df['WFO_ENC'] = le_wfo.fit_transform(df['WFO'].astype(str))

# Encode target labels (XGBoost wants integer classes)
le_y = LabelEncoder()
y_enc = le_y.fit_transform(df['EVENT_TYPE'])

# Stratified split (before imputation to prevent data leakage)
feature_cols = ['MONTH', 'HOUR', 'STATE_ENC', 'BEGIN_LAT', 'BEGIN_LON',
                'DURATION_MIN', 'CZ_TYPE_ENC', 'WFO_ENC']
X = df[feature_cols]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, stratify=y_enc, test_size=0.2, random_state=42
)

# --- Hierarchical lat/lon imputation (fit on training data only) ---
# Compute median coords at three geographic levels from training rows that have coords
train_df = df.loc[X_train.index]
has_coords = train_df[train_df['BEGIN_LAT'].notna()]
median_cz = has_coords.groupby(['STATE', 'CZ_FIPS'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_wfo = has_coords.groupby(['STATE', 'WFO'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_state = has_coords.groupby('STATE')[['BEGIN_LAT', 'BEGIN_LON']].median()

# Apply hierarchical fill to full df, then rebuild X
df_imp = df.copy()
missing = df_imp['BEGIN_LAT'].isna()

# Level 1: (STATE, CZ_FIPS) — county/zone median
for (state, cz_fips), row in median_cz.iterrows():
    mask = missing & (df_imp['STATE'] == state) & (df_imp['CZ_FIPS'] == cz_fips)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

missing = df_imp['BEGIN_LAT'].isna()

# Level 2: (STATE, WFO) — forecast office region median
for (state, wfo), row in median_wfo.iterrows():
    mask = missing & (df_imp['STATE'] == state) & (df_imp['WFO'] == wfo)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

missing = df_imp['BEGIN_LAT'].isna()

# Level 3: STATE — state median (last resort)
for state, row in median_state.iterrows():
    mask = missing & (df_imp['STATE'] == state)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

# Rebuild X from imputed data using the same indices
X_imp = df_imp[feature_cols]
X_train = X_imp.loc[X_train.index]
X_test = X_imp.loc[X_test.index]

# Sanity checks
print(f"\nShape: {X_train.shape}")
print(f"Missing per column (train):\n{X_train.isna().sum()}")
print(f"Missing per column (test):\n{X_test.isna().sum()}")

EVENT_TYPE
Thunderstorm Wind           41177
Hail                        20702
Drought                      9288
Flash Flood                  8286
Excessive Heat               7684
High Wind                    7485
Heat                         6494
Winter Weather               6448
Flood                        5318
Marine Thunderstorm Wind     4835
Name: count, dtype: int64

Shape: (94173, 8)
Missing per column (train):
MONTH           0
HOUR            0
STATE_ENC       0
BEGIN_LAT       0
BEGIN_LON       0
DURATION_MIN    0
CZ_TYPE_ENC     0
WFO_ENC         0
dtype: int64
Missing per column (test):
MONTH           0
HOUR            0
STATE_ENC       0
BEGIN_LAT       0
BEGIN_LON       0
DURATION_MIN    0
CZ_TYPE_ENC     0
WFO_ENC         0
dtype: int64


In [5]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Hyperparameter grid (proposal: learning rate, max depth, n_estimators, subsample)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'max_depth': [3, 5, 7, 9],
    'n_estimators': [100, 300, 500, 700],
    'subsample': [0.7, 0.8, 1.0],
}

base_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist',
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    verbose=1,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print(f"\nBest CV Macro F1: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

# Use the best model for evaluation
model = grid_search.best_estimator_

Fitting 5 folds for each of 192 candidates, totalling 960 fits


/Users/mdv/code/CISC484/noaa-storm-classification/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Best CV Macro F1: 0.8949
Best params: {'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 500, 'subsample': 0.7}


In [7]:
# Train default XGBoost for comparison
model_default = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist',
)
model_default.fit(X_train, y_train)
y_pred_default = model_default.predict(X_test)
macro_f1_default = f1_score(y_test, y_pred_default, average='macro')

# Evaluate tuned model on test set
y_pred_tuned = model.predict(X_test)
macro_f1_tuned = f1_score(y_test, y_pred_tuned, average='macro')

print("=" * 70)
print("Default XGBoost")
print("=" * 70)
print(f"Macro F1: {macro_f1_default:.4f}\n")
print(classification_report(y_test, y_pred_default, target_names=le_y.classes_))

print("\n" + "=" * 70)
print(f"Tuned XGBoost {grid_search.best_params_}")
print("=" * 70)
print(f"Macro F1: {macro_f1_tuned:.4f}\n")
print(classification_report(y_test, y_pred_tuned, target_names=le_y.classes_))

print("\n" + "=" * 70)
print(f"Improvement: {macro_f1_default:.4f} -> {macro_f1_tuned:.4f} ({macro_f1_tuned - macro_f1_default:+.4f})")
print("=" * 70)

Default XGBoost
Macro F1: 0.8887

                          precision    recall  f1-score   support

                 Drought       1.00      1.00      1.00      1858
          Excessive Heat       0.93      0.82      0.87      1537
             Flash Flood       0.87      0.87      0.87      1657
                   Flood       0.88      0.80      0.84      1064
                    Hail       0.74      0.65      0.69      4140
                    Heat       0.81      0.92      0.86      1299
               High Wind       0.96      0.95      0.96      1497
Marine Thunderstorm Wind       1.00      1.00      1.00       967
       Thunderstorm Wind       0.82      0.88      0.85      8235
          Winter Weather       0.95      0.96      0.95      1290

                accuracy                           0.86     23544
               macro avg       0.90      0.88      0.89     23544
            weighted avg       0.86      0.86      0.86     23544


Tuned XGBoost {'learning_rate': 0.1, '

In [8]:
# Confusion matrices side by side
cm_default = confusion_matrix(y_test, y_pred_default)
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

print("=" * 70)
print("Confusion Matrix — Default XGBoost")
print("=" * 70)
print(pd.DataFrame(cm_default, index=le_y.classes_, columns=le_y.classes_))

print("\n" + "=" * 70)
print("Confusion Matrix — Tuned XGBoost")
print("=" * 70)
print(pd.DataFrame(cm_tuned, index=le_y.classes_, columns=le_y.classes_))

# Highlight Hail <-> Thunderstorm Wind confusion (dominant error pattern)
print("\n" + "=" * 70)
print("Hail <-> Thunderstorm Wind confusion")
print("=" * 70)
hail_idx = list(le_y.classes_).index('Hail')
tsw_idx = list(le_y.classes_).index('Thunderstorm Wind')
print(f"  Default — Hail predicted as TStorm Wind: {cm_default[hail_idx, tsw_idx]}")
print(f"  Tuned   — Hail predicted as TStorm Wind: {cm_tuned[hail_idx, tsw_idx]}")
print(f"  Default — TStorm Wind predicted as Hail: {cm_default[tsw_idx, hail_idx]}")
print(f"  Tuned   — TStorm Wind predicted as Hail: {cm_tuned[tsw_idx, hail_idx]}")

# Feature importance comparison
imp_default = pd.Series(model_default.feature_importances_, index=feature_cols)
imp_tuned = pd.Series(model.feature_importances_, index=feature_cols)
imp_cmp = pd.DataFrame({
    'Default': imp_default,
    'Tuned': imp_tuned,
    'Diff': imp_tuned - imp_default,
}).sort_values('Default', ascending=False)

print("\n" + "=" * 70)
print("Feature Importance Comparison")
print("=" * 70)
print(imp_cmp.to_string(float_format='%.4f'))

Confusion Matrix — Default XGBoost
                          Drought  Excessive Heat  Flash Flood  Flood  Hail  \
Drought                      1853               3            0      0     0   
Excessive Heat                  3            1258            0      0     0   
Flash Flood                     0               0         1437     93    27   
Flood                           0               0          168    847    10   
Hail                            0               0           11      6  2692   
Heat                            4              98            0      0     0   
High Wind                       0               1            0      1     0   
Marine Thunderstorm Wind        0               0            0      0     0   
Thunderstorm Wind               0               0           31     13   927   
Winter Weather                  0               0            0      0     0   

                          Heat  High Wind  Marine Thunderstorm Wind  \
Drought                 

## Phase 2: TabNet

In [ ]:
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import StandardScaler

# TabNet requires standardized continuous features and no NaNs
# Standardize using training set statistics only (prevent data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

# Train default TabNet (sized for 8 features on CPU)
tabnet = TabNetClassifier(
    n_d=8,
    n_a=8,
    n_steps=3,
    gamma=1.3,
    optimizer_params=dict(lr=2e-2),
    seed=42,
    verbose=10,
)

tabnet.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    eval_metric=['accuracy'],
    max_epochs=50,
    patience=10,
    batch_size=8192,
    num_workers=0,
    drop_last=False,
)

In [ ]:
# Evaluate TabNet
y_pred_tabnet = tabnet.predict(X_test_scaled)
macro_f1_tabnet = f1_score(y_test, y_pred_tabnet, average='macro')

print("=" * 70)
print("Default TabNet")
print("=" * 70)
print(f"Macro F1: {macro_f1_tabnet:.4f}\n")
print(classification_report(y_test, y_pred_tabnet, target_names=le_y.classes_))

# Compare all three models
print("\n" + "=" * 70)
print("Summary: Default XGBoost vs Tuned XGBoost vs Default TabNet")
print("=" * 70)
print(f"  Default XGBoost:  {macro_f1_default:.4f}")
print(f"  Tuned XGBoost:    {macro_f1_tuned:.4f}")
print(f"  Default TabNet:   {macro_f1_tabnet:.4f}")

# Confusion matrix for TabNet
cm_tabnet = confusion_matrix(y_test, y_pred_tabnet)
print("\n" + "=" * 70)
print("Confusion Matrix — Default TabNet")
print("=" * 70)
print(pd.DataFrame(cm_tabnet, index=le_y.classes_, columns=le_y.classes_))

# Hail <-> Thunderstorm Wind comparison across all three
hail_idx = list(le_y.classes_).index('Hail')
tsw_idx = list(le_y.classes_).index('Thunderstorm Wind')
print("\n" + "=" * 70)
print("Hail <-> Thunderstorm Wind confusion (all models)")
print("=" * 70)
print(f"  Hail as TStorm Wind — Default XGB: {cm_default[hail_idx, tsw_idx]}, Tuned XGB: {cm_tuned[hail_idx, tsw_idx]}, TabNet: {cm_tabnet[hail_idx, tsw_idx]}")
print(f"  TStorm Wind as Hail — Default XGB: {cm_default[tsw_idx, hail_idx]}, Tuned XGB: {cm_tuned[tsw_idx, hail_idx]}, TabNet: {cm_tabnet[tsw_idx, hail_idx]}")